# 06: Multistream Pipeline

This notebook extends the work done in Notebook 01 and Notebook 02. It focuses on building a multistream data pipeline that can handle multiple synchronized streams from the VRS recording.

This next notebook will generalize that approach from one stream to all relevant streams.

## In this notebook, we will:

1. Inventory all streams
    - Build a canonical table with stream id, label, type, sample count, and time coverage.
    - Clearly separate image, IMU, and auxiliary streams.

2. Validate per-stream access
    - Confirm that each stream can be decoded or read by index.
    - Measure timestamp monotonicity and sample spacing per stream.
    - Flag empty, sparse, or non-standard streams.

3. Build synchronization and alignment logic
    - Select a reference stream (typically RGB) for alignment.
    - Align all streams to the reference using nearest-timestamp logic.
    - Produce a synchronization table for downstream use.

4. Define modular dataset abstractions
    - Use `AriaDataset` (from scripts) for single-stream access.
    - Implement `AriaMultistreamDataset` for synchronized multistream access, reusing the single-stream logic.
    - Ensure compatibility with PyTorch `Dataset` and `DataLoader`.

5. Add visualization and sanity checks
    - Display frame-aligned samples across multiple streams.
    - Inspect synchronization quality and data integrity.

6. Prepare for code extraction
    - Ensure all helpers and dataset classes are modular and ready to be moved to `scripts/`.
    - Keep the notebook focused on experimentation and validation, not on long-term implementation.

As a sample, we will use `kettle_and_forklift_recording.vrs` and its associated `kettle_and_forklift_recording.json` metadata, located in `data/raw/kettle_and_forklift`. The goal is to create a robust multistream pipeline that can be easily adapted to other recordings.

## 6.1 Setup and Recording Discovery

This section prepares the notebook for multistream work.
We reuse the same style established in Notebook 01 and Notebook 02:
- validate the recording paths
- make the project modules importable
- open the VRS provider
- discover the available streams as the first canonical inventory step


In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from projectaria_tools.core import data_provider, sensor_data


In [2]:
# Define the sample recording used for the multistream pipeline.
DATA_DIR = Path("..") / "data" / "raw" / "kettle_and_forklift"
VRS_PATH = DATA_DIR / "kettle_and_forklift_recording.vrs"
JSON_PATH = DATA_DIR / "kettle_and_forklift_recording.json"
OUT_DIR = Path("..") / "data" / "processed" / "kettle_and_forklift"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Validate the required input files before continuing.
if not VRS_PATH.exists():
    raise FileNotFoundError(f"VRS file not found: {VRS_PATH.resolve()}")
if not JSON_PATH.exists():
    raise FileNotFoundError(f"JSON metadata file not found: {JSON_PATH.resolve()}")

print("VRS path:", VRS_PATH.resolve())
print("JSON path:", JSON_PATH.resolve())
print("Output dir:", OUT_DIR.resolve())


VRS path: C:\Users\Gmalv\Desktop\Scuola_Universita\AriaGen1\data\raw\kettle_and_forklift\kettle_and_forklift_recording.vrs
JSON path: C:\Users\Gmalv\Desktop\Scuola_Universita\AriaGen1\data\raw\kettle_and_forklift\kettle_and_forklift_recording.json
Output dir: C:\Users\Gmalv\Desktop\Scuola_Universita\AriaGen1\data\processed\kettle_and_forklift


In [ ]:
# Make the repository root importable so helper modules in aria_pylib can be reused.
project_root = next((p for p in [Path.cwd().resolve(), Path.cwd().resolve().parent] if (p / "aria_pylib" / "__init__.py").exists()), None)
if project_root is None:
    raise RuntimeError("Could not find the project root containing aria_pylib/")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Reuse the frame extraction helper established in aria_pylib.
from aria_pylib import _extract_image_array_from_sample

# Create the VRS provider used by the rest of the notebook.
provider = data_provider.create_vrs_data_provider(str(VRS_PATH))
if provider is None:
    raise RuntimeError("Failed to initialize the VRS data provider.")

print("Provider initialized successfully.")
print("Available streams:", len(provider.get_all_streams()))

Provider initialized successfully.
Available streams: 11


## 6.2 Stream Inventory and Canonical Table

This section recreates the stream discovery pattern from Notebook 01, but with a multistream perspective.
We want one canonical table that makes the recording structure explicit before any dataset or synchronization logic is introduced.

The table will be the base artifact for the rest of the notebook, every later decision should refer back to this.

In [4]:
# Helper to classify streams into a broad modality category.
def infer_modality(label: str) -> str:
    """Infer a broad modality label from the stream name or label."""
    text = (label or "").lower()
    if any(key in text for key in ["camera", "image", "rgb", "slam", "et"]):
        return "image"
    if any(key in text for key in ["imu", "accel", "gyro"]):
        return "imu"
    if any(key in text for key in ["audio", "mic"]):
        return "audio"
    if any(key in text for key in ["gps", "baro", "mag", "bluetooth", "ble", "wifi", "wps"]):
        return "auxiliary"
    return "unknown"


In [9]:
# Build a canonical inventory table for all available streams.
stream_rows = []
for stream_id in provider.get_all_streams():
    stream_label = provider.get_label_from_stream_id(stream_id)
    sample_count = provider.get_num_data(stream_id)
    stream_rows.append({
        "stream_id": str(stream_id),
        "label": str(stream_label),
        "type": infer_modality(str(stream_label)),
        "sample_count": int(sample_count),
    })

streams_df = pd.DataFrame(stream_rows).sort_values(["type", "label", "stream_id"]).reset_index(drop=True)

print(f"Discovered {len(streams_df)} streams.")
display(streams_df)

Discovered 11 streams.


,stream_id,label,type,sample_count
0,247-1,baro0,auxiliary,2565
1,281-1,gps,auxiliary,52
2,1203-1,mag0,auxiliary,516
3,282-1,wps,auxiliary,136
4,283-1,bluetooth,image,0
5,211-1,camera-et,image,1551
6,214-1,camera-rgb,image,1551
7,1201-1,camera-slam-left,image,1551
8,1201-2,camera-slam-right,image,1551
9,1202-2,imu-left,imu,41675


In [10]:
# Keep a short summary that can be reused by later sections.
stream_summary_df = streams_df.groupby("type", dropna=False).size().reset_index(name="num_streams")
display(stream_summary_df)

,type,num_streams
0,auxiliary,4
1,image,5
2,imu,2


## 6.3 MultiStream Timestamp Validation

The next step is to validate access and timing for each stream in a reusable way.


In [11]:
# Helper to resolve a stream identifier back to the provider stream object.
def resolve_stream_id(provider_obj, stream_id_str: str):
    """Resolve a string stream id back to the provider-native stream object."""
    matches = [sid for sid in provider_obj.get_all_streams() if str(sid) == str(stream_id_str)]
    if len(matches) == 0:
        return None
    return matches[0]


# Helper to read timestamps for one stream using index-based access.
def get_stream_timestamps_ns(provider_obj, stream_id_obj):
    """Read all DEVICE_TIME timestamps for a stream as a NumPy int64 array."""
    sample_count = int(provider_obj.get_num_data(stream_id_obj))
    timestamps_ns = np.empty(sample_count, dtype=np.int64)
    for index in range(sample_count):
        sample = provider_obj.get_sensor_data_by_index(stream_id_obj, index)
        timestamps_ns[index] = sample.get_time_ns(sensor_data.TimeDomain.DEVICE_TIME)
    return timestamps_ns


# Helper to compute a compact timing summary from a timestamp array.
def summarize_timestamp_series(timestamps_ns):
    """Compute timing statistics for one timestamp series."""
    timestamps_ns = np.asarray(timestamps_ns, dtype=np.int64)
    if timestamps_ns.size == 0:
        return {
            "sample_count": 0,
            "t_min_ns": None,
            "t_max_ns": None,
            "duration_s": 0.0,
            "delta_min_ms": None,
            "delta_median_ms": None,
            "delta_max_ms": None,
            "strictly_increasing": True,
        }

    deltas_ns = np.diff(timestamps_ns)
    positive_deltas_ns = deltas_ns[deltas_ns > 0]
    return {
        "sample_count": int(timestamps_ns.size),
        "t_min_ns": int(timestamps_ns.min()),
        "t_max_ns": int(timestamps_ns.max()),
        "duration_s": float((timestamps_ns.max() - timestamps_ns.min()) / 1e9) if timestamps_ns.size > 1 else 0.0,
        "delta_min_ms": float(positive_deltas_ns.min() / 1e6) if positive_deltas_ns.size > 0 else None,
        "delta_median_ms": float(np.median(positive_deltas_ns) / 1e6) if positive_deltas_ns.size > 0 else None,
        "delta_max_ms": float(positive_deltas_ns.max() / 1e6) if positive_deltas_ns.size > 0 else None,
        "strictly_increasing": bool(np.all(deltas_ns > 0)) if deltas_ns.size > 0 else True,
    }


# Build the per-stream timing report for the full inventory.
timestamp_rows = []
for _, stream_row in streams_df.iterrows():
    stream_id_obj = resolve_stream_id(provider, stream_row["stream_id"])
    if stream_id_obj is None:
        timestamp_rows.append({
            "stream_id": stream_row["stream_id"],
            "label": stream_row["label"],
            "type": stream_row["type"],
            "sample_count": 0,
            "t_min_ns": None,
            "t_max_ns": None,
            "duration_s": None,
            "delta_min_ms": None,
            "delta_median_ms": None,
            "delta_max_ms": None,
            "strictly_increasing": False,
            "status": "UNRESOLVABLE",
        })
        continue

    try:
        timestamps_ns = get_stream_timestamps_ns(provider, stream_id_obj)
        timing_summary = summarize_timestamp_series(timestamps_ns)
        timing_summary.update({
            "stream_id": stream_row["stream_id"],
            "label": stream_row["label"],
            "type": stream_row["type"],
            "status": "OK",
        })
        timestamp_rows.append(timing_summary)
    except Exception as exc:
        timestamp_rows.append({
            "stream_id": stream_row["stream_id"],
            "label": stream_row["label"],
            "type": stream_row["type"],
            "sample_count": int(stream_row["sample_count"]),
            "t_min_ns": None,
            "t_max_ns": None,
            "duration_s": None,
            "delta_min_ms": None,
            "delta_median_ms": None,
            "delta_max_ms": None,
            "strictly_increasing": False,
            "status": f"ERROR: {exc}",
        })

# Build and format the DataFrame with a stable, explicit column order.
stream_timing_df = pd.DataFrame(timestamp_rows)
stream_timing_df = stream_timing_df.sort_values(["status", "type", "label", "stream_id"]).reset_index(drop=True)

ordered_columns = [
    "stream_id",
    "label",
    "type",
    "sample_count",
    "t_min_ns",
    "t_max_ns",
    "duration_s",
    "delta_min_ms",
    "delta_median_ms",
    "delta_max_ms",
    "strictly_increasing",
    "status",
]
stream_timing_df = stream_timing_df[ordered_columns]

display(stream_timing_df)

,stream_id,label,type,sample_count,t_min_ns,t_max_ns,duration_s,delta_min_ms,delta_median_ms,delta_max_ms,strictly_increasing,status
0,247-1,baro0,auxiliary,2565,8.883402e+11,9.400196e+11,51.679389,19.325863,20.159750,20.978412,True,OK
1,281-1,gps,auxiliary,52,8.886455e+11,9.396473e+11,51.001845,994.000677,999.652656,1005.927447,True,OK
2,1203-1,mag0,auxiliary,516,8.884287e+11,9.399679e+11,51.539196,100.007125,100.084300,100.161875,True,OK
3,282-1,wps,auxiliary,136,-1.000000e+00,-1.000000e+00,0.000000,NaN,NaN,NaN,False,OK
4,283-1,bluetooth,image,0,NaN,NaN,0.000000,NaN,NaN,NaN,True,OK
5,211-1,camera-et,image,1551,8.883627e+11,9.400211e+11,51.658400,33.328000,33.328000,33.328000,True,OK
6,214-1,camera-rgb,image,1551,8.883626e+11,9.400210e+11,51.658395,33.037587,33.327338,33.614500,True,OK
7,1201-1,camera-slam-left,image,1551,8.883627e+11,9.400211e+11,51.658400,33.328000,33.328000,33.328000,True,OK
8,1201-2,camera-slam-right,image,1551,8.883627e+11,9.400211e+11,51.658400,33.328000,33.328000,33.328000,True,OK
9,1202-2,imu-left,imu,41675,8.883457e+11,9.400311e+11,51.685367,1.240050,1.240238,1.240413,True,OK


### Interpretation

This table is the first quality gate for the multistream pipeline.
It tells us which streams are valid for downstream use, which streams are empty or sparse, and whether the timestamps behave as expected.

Once this report is stable, we can safely move on to stream-level integrity checks and dataset design.

### Note on sparse auxiliary streams

It is normal for `wps` and `bluetooth` to show many `NaN` values in later timing or quality tables.
Especially `bluetooth` was turned off during the recording, so it is expected to be empty.
For that reason, they should be treated as auxiliary streams and analyzed separately from the main synchronized recording streams.

## 6.4 Sample Read Integrity Checks

This section checks a few representative indices per stream.
The goal is not to exhaustively test every sample, but to confirm that index-based access works consistently across the inventory.

### 6.4.1 Integrity Check Strategy

We use a lightweight probing strategy to validate per-stream readability:
- resolve stream id from the canonical table
- test a small set of representative indices
- report pass/fail counts and a compact status

This gives a robust sanity check without scanning every sample.

In [12]:
# Helper to select representative sample indices for quick integrity probing.
def select_probe_indices(sample_count: int):
    """Select a compact set of representative indices for one stream."""
    if sample_count <= 0:
        return []

    candidate_indices = [
        0,
        sample_count // 4,
        sample_count // 2,
        (3 * sample_count) // 4,
        sample_count - 1,
    ]

    # Remove duplicates while preserving order.
    probe_indices = []
    seen = set()
    for idx in candidate_indices:
        idx = int(idx)
        if idx not in seen:
            probe_indices.append(idx)
            seen.add(idx)
    return probe_indices


# Helper to probe index-based readability for one stream.
def probe_stream_readability(provider_obj, stream_id_obj, probe_indices):
    """Attempt to read a list of indices and return integrity counters."""
    read_ok = 0
    read_fail = 0
    first_error = None

    for idx in probe_indices:
        try:
            _ = provider_obj.get_sensor_data_by_index(stream_id_obj, int(idx))
            read_ok += 1
        except Exception as exc:
            read_fail += 1
            if first_error is None:
                first_error = str(exc)

    if read_fail == 0:
        status = "OK"
    elif read_ok > 0:
        status = "PARTIAL"
    else:
        status = "ERROR"

    return {
        "tested_indices": probe_indices,
        "read_ok": read_ok,
        "read_fail": read_fail,
        "status": status,
        "error_example": first_error,
    }

In [13]:
# Build the stream integrity report using the modular probe helpers.
integrity_rows = []
for _, stream_row in streams_df.iterrows():
    stream_id_str = stream_row["stream_id"]
    stream_id_obj = resolve_stream_id(provider, stream_id_str)
    sample_count = int(stream_row["sample_count"])

    # Handle unresolved stream ids explicitly.
    if stream_id_obj is None:
        integrity_rows.append({
            "stream_id": stream_id_str,
            "label": stream_row["label"],
            "type": stream_row["type"],
            "sample_count": sample_count,
            "tested_indices": [],
            "read_ok": 0,
            "read_fail": 0,
            "status": "UNRESOLVABLE",
            "error_example": "Stream id not resolvable from provider.",
        })
        continue

    # Handle empty streams as a valid special case.
    if sample_count == 0:
        integrity_rows.append({
            "stream_id": stream_id_str,
            "label": stream_row["label"],
            "type": stream_row["type"],
            "sample_count": 0,
            "tested_indices": [],
            "read_ok": 0,
            "read_fail": 0,
            "status": "EMPTY",
            "error_example": None,
        })
        continue

    probe_indices = select_probe_indices(sample_count)
    probe_result = probe_stream_readability(provider, stream_id_obj, probe_indices)

    integrity_rows.append({
        "stream_id": stream_id_str,
        "label": stream_row["label"],
        "type": stream_row["type"],
        "sample_count": sample_count,
        "tested_indices": probe_result["tested_indices"],
        "read_ok": probe_result["read_ok"],
        "read_fail": probe_result["read_fail"],
        "status": probe_result["status"],
        "error_example": probe_result["error_example"],
    })

stream_integrity_df = pd.DataFrame(integrity_rows)
stream_integrity_df = stream_integrity_df.sort_values(["status", "type", "label", "stream_id"]).reset_index(drop=True)

display(stream_integrity_df)

# Also expose a compact status summary for quick inspection.
integrity_status_df = stream_integrity_df.groupby("status", dropna=False).size().reset_index(name="num_streams")
display(integrity_status_df)

,stream_id,label,type,sample_count,tested_indices,read_ok,read_fail,status,error_example
0,283-1,bluetooth,image,0,[],0,0,EMPTY,None
1,247-1,baro0,auxiliary,2565,"[0, 641, 1282, 1923, 2564]",5,0,OK,None
2,281-1,gps,auxiliary,52,"[0, 13, 26, 39, 51]",5,0,OK,None
3,1203-1,mag0,auxiliary,516,"[0, 129, 258, 387, 515]",5,0,OK,None
4,282-1,wps,auxiliary,136,"[0, 34, 68, 102, 135]",5,0,OK,None
5,211-1,camera-et,image,1551,"[0, 387, 775, 1163, 1550]",5,0,OK,None
6,214-1,camera-rgb,image,1551,"[0, 387, 775, 1163, 1550]",5,0,OK,None
7,1201-1,camera-slam-left,image,1551,"[0, 387, 775, 1163, 1550]",5,0,OK,None
8,1201-2,camera-slam-right,image,1551,"[0, 387, 775, 1163, 1550]",5,0,OK,None
9,1202-2,imu-left,imu,41675,"[0, 10418, 20837, 31256, 41674]",5,0,OK,None


,status,num_streams
0,EMPTY,1
1,OK,10


### 6.4.2 Integrity Results Interpretation

Interpretation guidelines:
- `OK`: all probed indices were read successfully.
- `PARTIAL`: only a subset of probed indices was readable.
- `ERROR`: all probes failed for that stream.
- `EMPTY`: the stream has no samples in this recording.
- `UNRESOLVABLE`: stream id could not be mapped back to a provider stream object.

This report complements the timestamp validation from Section 6.3 and forms the second quality gate before synchronization.

## 6.5 Stream Synchronization and Alignment

This section introduces the logic and helpers for synchronizing multiple streams on a common time axis.

The goal is to enable robust, modular alignment of streams with different rates, missing samples, and partial overlaps.

We will:
- Define a reference stream (we will use the RGB stream) to serve as the alignment anchor.
- Build helpers to align other streams to this reference, handling missing data and rate mismatches.
- Produce a synchronization table that can be reused for dataset construction and downstream analysis.

In [15]:
# Helper to select the RGB stream as the reference for synchronization.
def select_rgb_reference_stream(streams_df):
    """Select the first stream whose label contains 'rgb' (case-insensitive)."""
    rgb_candidates = streams_df[streams_df["label"].str.lower().str.contains("rgb")]
    if not rgb_candidates.empty:
        return rgb_candidates.iloc[0]["stream_id"]
    # Fallback: use the first image stream if no explicit RGB found.
    image_candidates = streams_df[streams_df["type"] == "image"]
    if not image_candidates.empty:
        return image_candidates.iloc[0]["stream_id"]
    # Fallback: use the first available stream.
    return streams_df.iloc[0]["stream_id"] if not streams_df.empty else None

# Helper to build a synchronization table aligning all streams to the reference stream.
def build_synchronization_table(provider, streams_df, reference_stream_id):
    """Align all streams to the reference stream by nearest timestamp."""
    ref_stream_obj = resolve_stream_id(provider, reference_stream_id)
    ref_timestamps = get_stream_timestamps_ns(provider, ref_stream_obj)
    sync_rows = []
    for _, stream_row in streams_df.iterrows():
        stream_id = stream_row["stream_id"]
        stream_obj = resolve_stream_id(provider, stream_id)
        if stream_obj is None:
            sync_rows.append({
                "stream_id": stream_id,
                "label": stream_row["label"],
                "type": stream_row["type"],
                "aligned_indices": [],
                "aligned_timestamps_ns": [],
                "num_aligned": 0,
                "status": "UNRESOLVABLE",
            })
            continue
        stream_timestamps = get_stream_timestamps_ns(provider, stream_obj)
        # For each reference timestamp, find the closest timestamp in this stream.
        aligned_indices = []
        aligned_timestamps = []
        for t_ref in ref_timestamps:
            if len(stream_timestamps) == 0:
                idx = None
                t_aligned = None
            else:
                idx = int(np.argmin(np.abs(stream_timestamps - t_ref)))
                t_aligned = int(stream_timestamps[idx])
            aligned_indices.append(idx)
            aligned_timestamps.append(t_aligned)
        sync_rows.append({
            "stream_id": stream_id,
            "label": stream_row["label"],
            "type": stream_row["type"],
            "aligned_indices": aligned_indices,
            "aligned_timestamps_ns": aligned_timestamps,
            "num_aligned": len(aligned_indices),
            "status": "OK" if stream_obj is not None else "UNRESOLVABLE",
        })
    sync_df = pd.DataFrame(sync_rows)
    return sync_df

# select the RGB reference stream and build the synchronization table.
reference_stream_id = select_rgb_reference_stream(streams_df)
print(f"Selected RGB reference stream: {reference_stream_id}")
sync_df = build_synchronization_table(provider, streams_df, reference_stream_id)
# Show all relevant columns for interpretation, not just the summary.
display(sync_df[["stream_id", "label", "type", "num_aligned", "status", "aligned_indices", "aligned_timestamps_ns"]])

Selected RGB reference stream: 214-1


,stream_id,label,type,num_aligned,status,aligned_indices,aligned_timestamps_ns
0,247-1,baro0,auxiliary,1551,OK,"[1, 3, 4, 6, 8, 9, 11, 13, 14, 16, 18, 19, 21,...","[888360387275, 888400703900, 888420859875, 888..."
1,281-1,gps,auxiliary,1551,OK,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[888645454450, 888645454450, 888645454450, 888..."
2,1203-1,mag0,auxiliary,1551,OK,"[0, 0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, ...","[888428738900, 888428738900, 888428738900, 888..."
3,282-1,wps,auxiliary,1551,OK,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -..."
4,283-1,bluetooth,image,1551,OK,"[None, None, None, None, None, None, None, Non...","[None, None, None, None, None, None, None, Non..."
5,211-1,camera-et,image,1551,OK,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[888362705312, 888396033312, 888429361312, 888..."
6,214-1,camera-rgb,image,1551,OK,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[888362615137, 888395942475, 888429273512, 888..."
7,1201-1,camera-slam-left,image,1551,OK,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[888362708812, 888396036812, 888429364812, 888..."
8,1201-2,camera-slam-right,image,1551,OK,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[888362708800, 888396036800, 888429364800, 888..."
9,1202-2,imu-left,imu,1551,OK,"[14, 40, 67, 94, 121, 148, 175, 202, 229, 255,...","[888363088100, 888395330150, 888428812300, 888..."


### Synchronization Table Interpretation

The synchronization table summarizes how each stream is aligned to the reference stream.

- `aligned_indices` and `aligned_timestamps_ns` show, for each reference sample, the closest sample in the target stream.
- `num_aligned` gives the number of reference points for which an alignment was found.
- `status` indicates whether the stream was successfully aligned or not.

This table is the foundation for building a robust multistream dataset, enabling frame-accurate sampling and analysis across all streams.

## 6.6 Multistream Dataset Abstraction

In this section, we will define a modular Python class to represent a synchronized multistream dataset.

The goal is to provide a reusable, PyTorch-style interface that returns a dictionary of synchronized samples for each index, using the alignment table built in the previous step.

Key features:
- Each item is a dictionary mapping stream labels to decoded samples aligned to the reference RGB frame.
- Handles missing data gracefully (returns None or a placeholder if a stream is not available at a given index).

We will first define the class, then show example usage and inspection.

### 6.6.1 Modular Multistream Dataset: AriaMultistreamDataset

We now define the multistream dataset as a natural extension of the single-stream `AriaDataset` (imported from `scripts/aria_dataset.py`).

- `AriaDataset` is used for single-stream, index-based access.
- `AriaMultistreamDataset` inherits from `AriaDataset`, reusing its logic for per-stream access, and adds synchronization logic to return a dictionary of aligned samples.

This approach ensures code reuse, maintainability, and a clear upgrade path from single-stream to multistream pipelines.

Below, we define the modular multistream class and show example usage.

In [17]:
# Import the single-stream AriaDataset from scripts/aria_dataset.py
from scripts.aria_dataset import AriaDataset
from typing import Dict, Any, Optional

# the Multistream dataset Inherits from AriaDataset 
class AriaMultistreamDataset(AriaDataset):
    """A modular dataset returning synchronized samples from multiple streams, aligned to the reference stream."""
    def __init__(self, provider, streams_df, sync_df, decode_image_fn=None):
        # Find the reference stream (RGB or first image stream)
        ref_row = sync_df[sync_df["label"].str.lower().str.contains("rgb")].head(1)
        if ref_row.empty:
            ref_row = sync_df[sync_df["type"] == "image"].head(1)
        if not ref_row.empty:
            ref_stream_id = ref_row.iloc[0]["stream_id"]
            ref_indices = ref_row.iloc[0]["aligned_indices"]
        else:
            ref_stream_id = None
            ref_indices = []
        # Use the base class for reference stream access
        ref_timestamps_ns = []
        if ref_stream_id is not None and len(ref_indices) > 0:
            # Get the timestamps for the reference stream
            ref_stream_obj = resolve_stream_id(provider, ref_stream_id)
            ref_timestamps_ns = get_stream_timestamps_ns(provider, ref_stream_obj)
        super().__init__(provider, ref_stream_id, ref_timestamps_ns)
        self.streams_df = streams_df
        self.sync_df = sync_df
        self.decode_image_fn = decode_image_fn or (lambda sample: sample)
        self.stream_id_to_label = dict(zip(streams_df["stream_id"], streams_df["label"]))
        self.length = len(ref_indices)
        self.ref_indices = ref_indices

    def __len__(self):
        return self.length

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        sample = {}
        for _, row in self.sync_df.iterrows():
            label = row["label"]
            indices = row["aligned_indices"]
            if idx >= len(indices) or indices[idx] is None or indices[idx] == -1:
                sample[label] = None
                continue
            stream_id = row["stream_id"]
            stream_obj = resolve_stream_id(self.provider, stream_id)
            try:
                raw_sample = self.provider.get_sensor_data_by_index(stream_obj, indices[idx])
                if row["type"] == "image":
                    sample[label] = self.decode_image_fn(raw_sample)
                else:
                    sample[label] = raw_sample
            except Exception:
                sample[label] = None
        return sample

In [18]:
# example
multistream_dataset = AriaMultistreamDataset(provider, streams_df, sync_df, decode_image_fn=_extract_image_array_from_sample)
print(f"Multistream dataset length: {len(multistream_dataset)}")
example_sample = multistream_dataset[0]
for k, v in example_sample.items():
    print(f"{k}: {type(v)}")


Multistream dataset length: 1551
baro0: <class '_core_pybinds.sensor_data.SensorData'>
gps: <class '_core_pybinds.sensor_data.SensorData'>
mag0: <class '_core_pybinds.sensor_data.SensorData'>
wps: <class '_core_pybinds.sensor_data.SensorData'>
bluetooth: <class 'NoneType'>
camera-et: <class 'numpy.ndarray'>
camera-rgb: <class 'numpy.ndarray'>
camera-slam-left: <class 'numpy.ndarray'>
camera-slam-right: <class 'numpy.ndarray'>
imu-left: <class '_core_pybinds.sensor_data.SensorData'>
imu-right: <class '_core_pybinds.sensor_data.SensorData'>


## Conclusion

This notebook demonstrated a robust, modular approach for synchronized multistream data access from Project Aria VRS recordings.

- All relevant streams were inventoried, validated, and synchronized to a common reference.
- The modular `AriaMultistreamDataset` enables frame-accurate, dictionary-based access to all streams, ready for PyTorch pipelines.
- The design is cleanly separated for easy extraction into reusable scripts or packages.

This foundation supports scalable, maintainable research and development on Aria multistream data.